# Categorical Data

## Definition
The **Categorical** dtype in pandas stores data with a **limited, fixed set of values** (categories).
It's optimal for columns like gender, department, grade, city, etc.

### Benefits
- **Memory efficient**: stores integer codes instead of strings
- **Faster operations**: comparisons, groupby, sorting
- **Ordered categories**: allows `<`, `>` comparisons between categories
- **Explicit missing category**: `pd.NA` category

### String (object) vs Categorical Memory
```
1M rows of 'Male'/'Female' strings:  ~64 MB
1M rows as categorical:               ~1 MB  ← 64x more efficient!
```

### Key Operations
| Operation | Code |
|-----------|------|
| Convert to categorical | `df['col'].astype('category')` |
| Get categories | `df['col'].cat.categories` |
| Get codes | `df['col'].cat.codes` |
| Add category | `df['col'].cat.add_categories(...)` |
| Remove unused | `df['col'].cat.remove_unused_categories()` |
| Set ordered | `pd.Categorical(..., ordered=True)` |

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'name':  ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'dept':  ['Engineering', 'Marketing', 'Engineering', 'HR', 'Marketing'],
    'grade': ['A', 'C', 'B', 'D', 'A'],
    'size':  ['Large', 'Small', 'Medium', 'Small', 'Large']
})

# Convert to categorical
df['dept'] = df['dept'].astype('category')
df['grade'] = df['grade'].astype('category')
print(df.dtypes)

In [ ]:
# Inspect categories
print('dept categories:', df['dept'].cat.categories.tolist())
print('dept codes:', df['dept'].cat.codes.values)
print()
print('grade categories:', df['grade'].cat.categories.tolist())
print('grade codes:', df['grade'].cat.codes.values)

In [ ]:
# Memory comparison
n = 1_000_000
obj_series = pd.Series(['Engineering', 'Marketing', 'HR'] * (n//3))
cat_series = obj_series.astype('category')

print(f'Object dtype memory: {obj_series.memory_usage(deep=True)/1024/1024:.1f} MB')
print(f'Category dtype memory: {cat_series.memory_usage(deep=True)/1024/1024:.1f} MB')

In [ ]:
# Ordered categories (for grades, sizes)
from pandas.api.types import CategoricalDtype

# Define ordered categories
size_order = CategoricalDtype(['Small', 'Medium', 'Large', 'XLarge'], ordered=True)
df['size'] = df['size'].astype(size_order)
print('Ordered size categories:', df['size'].cat.categories.tolist())
print('Is ordered:', df['size'].cat.ordered)

# Now we can use comparisons!
print('Small shirts:', df[df['size'] < 'Large'][['name', 'size']])

In [ ]:
# Ordered grade categories for filtering
grade_order = CategoricalDtype(['F', 'D', 'C', 'B', 'A'], ordered=True)
df['grade_ord'] = df['grade'].astype(grade_order)
print('Grade B or better:')
print(df[df['grade_ord'] >= 'B'][['name', 'grade_ord']])

In [ ]:
# Categorical in groupby (includes empty categories)
cat = pd.Categorical(['A', 'B', 'A', 'A'], categories=['A', 'B', 'C'])
s = pd.Series(cat)
print('Value counts (includes empty C):')
print(s.value_counts())

# groupby observed=False includes all categories
df_cat = pd.DataFrame({'cat': s, 'val': [10, 20, 30, 40]})
print('\nGroupby with observed=False:')
print(df_cat.groupby('cat', observed=False)['val'].sum())

In [ ]:
# add/remove categories
df2 = df.copy()
df2['dept'] = df2['dept'].cat.add_categories(['Finance', 'Legal'])
print('After adding categories:', df2['dept'].cat.categories.tolist())

# Remove unused categories
df2['dept'] = df2['dept'].cat.remove_unused_categories()
print('After removing unused:', df2['dept'].cat.categories.tolist())